[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mauriciorslrv/DS_basics/blob/main/03%20Aprendizaje%20M%C3%A1quina/notebooks/03_Preparar_Datos_para_ML.ipynb)

# 03 · Preparar datos para ML

> **Pregunta guía:** antes de una hora de operación, ¿qué información permitiría anticipar la demanda de bicicletas?

Usaremos **Bike Sharing** de Capital Bikeshare, Washington, D. C., publicado por Fanaee-T y Gama. Reúne registros horarios de 2011–2012: cada fila representa una hora y **cnt** es el total de alquileres. UCI distribuye el dataset con licencia CC BY 4.0.

**Objetivo:** entender la tabla y preparar una evaluación realista antes de entrenar: definir X/y, identificar columnas inválidas, respetar el orden temporal y ajustar transformaciones sólo con entrenamiento.

📊 [Ficha UCI, atribución y licencia](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset) · DOI 10.24432/C5W894.

## 1. Cargar el dataset

Descargamos la versión horaria directamente de UCI al ejecutar. No se inventan observaciones ni se guarda una copia local; así el ejercicio usa los registros documentados por la fuente.

In [ ]:
# Datos horarios reales de UCI Bike Sharing; añadimos cnt como objetivo para estudiar preparación de datos.
%pip -q install ucimlrepo

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo

bike = fetch_ucirepo(id=275)
X_original = bike.data.features.copy()
y_original = bike.data.targets.copy()
df = X_original.copy()
df["cnt"] = y_original.iloc[:, 0].to_numpy()
df.head()

## 2. Entender la tabla antes de modelar

Una fila representa una hora, no una persona ni un viaje individual. Revisamos tipos, faltantes, duplicados y rangos porque un modelo puede ejecutarse aunque la tabla esté mal interpretada. Consulta la definición de cada campo en UCI; los nombres abreviados no bastan para inferir su significado.

In [ ]:
print("Dimensiones:", df.shape)
print("Columnas:", df.columns.tolist())
display(df.dtypes.to_frame("tipo"))
display(df.isna().sum().sort_values(ascending=False).head(10).to_frame("faltantes"))
print("Filas duplicadas:", df.duplicated().sum())
display(df.describe(include="all").T)

### Visualización: ¿cómo se distribuye la demanda?

Un histograma muestra la forma de la variable objetivo. La figura sirve para hacer una pregunta sobre los datos; no basta con mostrarla y seguir.

In [ ]:
df["cnt"].hist(bins=40, color="#176b87", edgecolor="white")
plt.title("Alquileres de bicicletas por hora")
plt.xlabel("Alquileres (cnt)")
plt.ylabel("Número de horas")
plt.show()

## 3. Definir el objetivo y reconocer fuga de información

El objetivo **cnt** es el número total de alquileres en esa hora. **casual** y **registered** desglosan exactamente ese total; incluirlas equivale a entregar partes de la respuesta. **instant** es un identificador, no una condición descriptiva útil por sí misma.

**dteday** se conserva para ordenar y separar periodos, no entra como predictor inicial. Las demás columnas candidatas describen calendario u otras condiciones registradas (hora, día, clima). Se incluyen para preguntar qué señal aportan en este dataset.

Hay un límite importante: el clima observado en la misma hora quizá no se conozca al emitir un pronóstico. Para anticipar de verdad, habría que usar el pronóstico meteorológico que estaba disponible entonces o excluir esas variables. La validez depende del momento de predicción, no sólo de que una columna esté en la tabla.

In [ ]:
target = "cnt"
leakage_and_id = [c for c in ["casual", "registered", "instant"] if c in df.columns]
feature_cols = [c for c in df.columns if c not in [target, *leakage_and_id, "dteday"]]

print("Excluidas:", leakage_and_id)
print("Predictores candidatos:", feature_cols)
if {"casual", "registered", "cnt"}.issubset(df.columns):
    print("¿casual + registered = cnt en todas las filas?",
          (df["casual"] + df["registered"] == df["cnt"]).all())

## 4. Separar entrenamiento y prueba por fecha

Como la pregunta es sobre horas futuras, ordenamos por **dteday** y reservamos las fechas más recientes para prueba. El corte se hace entre fechas, no en medio de un día. Mezclar aleatoriamente registros de 2011 y 2012 podría poner patrones casi contemporáneos en ambos grupos y exagerar el rendimiento futuro.

In [ ]:
df["dteday"] = pd.to_datetime(df["dteday"])
unique_dates = np.sort(df["dteday"].dropna().unique())
cutoff = unique_dates[int(0.80 * len(unique_dates))]

train = df[df["dteday"] < cutoff].copy()
test = df[df["dteday"] >= cutoff].copy()
X_train, y_train = train[feature_cols], train[target]
X_test, y_test = test[feature_cols], test[target]

print("Corte:", pd.Timestamp(cutoff).date())
print("Train:", X_train.shape, train["dteday"].min().date(), "→", train["dteday"].max().date())
print("Test:", X_test.shape, test["dteday"].min().date(), "→", test["dteday"].max().date())

## 5. Preprocesar dentro de un pipeline

Los campos de calendario y códigos de clima se tratan como categorías: aunque tengan números, sus valores representan etiquetas y no necesariamente distancias. Las mediciones se tratan como numéricas. El imputador, escalador y codificador aprenden sus parámetros sólo en entrenamiento; luego transforman test con esas reglas.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
for col in ["season", "yr", "mnth", "hr", "holiday", "weekday", "workingday", "weathersit"]:
    if col in X_train.columns and col not in categorical:
        categorical.append(col)
numeric = [c for c in X_train.columns if c not in categorical]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])
category_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric),
    ("cat", category_pipe, categorical)
])

X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)
print("Numéricas:", numeric)
print("Categóricas:", categorical)
print("Matrices:", X_train_ready.shape, X_test_ready.shape)

## 🧠 Qué aprendimos

- Primero se define la unidad de análisis y el significado de la etiqueta.
- **casual** y **registered** revelan componentes de **cnt**; **instant** sólo identifica la fila.
- **dteday** ordena la evaluación, pero no se usa aquí como predictor.
- Una predicción futura sólo puede usar información disponible en ese momento.
- El test temporal simula un periodo posterior y las transformaciones se aprenden en entrenamiento.

## 🧪 Mini reto

Mueve el corte temporal al 70 % y al 90 %. Explica cuántas fechas quedan para entrenar y evaluar. Después plantea dos escenarios, uno con pronóstico del clima y otro sin clima: ¿qué columnas cambiarían y por qué?

## Fuentes y atribución

- Fanaee-T, H. & Gama, J. (2013). UCI Bike Sharing Dataset, basado en registros de Capital Bikeshare en Washington, D. C. [Ficha, variables, atribución y licencia CC BY 4.0](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset), DOI 10.24432/C5W894.
- Scikit-learn: [Pipelines y composición](https://scikit-learn.org/stable/modules/compose.html) y [errores comunes](https://scikit-learn.org/stable/common_pitfalls.html).